In [1]:
import pandas as pd
from pathlib import Path
from functools import reduce
import numpy as np

In [2]:
# Load the dataset created in the data collection step
processed_dir = Path("D:\Erdos\Data Science Bootcamp\early_diabetes_screening\data\processed")
df_project = pd.read_csv(processed_dir / "project_dataset.csv")
print(df_project.shape)
df_project.head()

(15560, 35)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPXOSY1,BPXOSY2,...,PAD675,PAD680,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910
0,109263.0,2.0,1.0,6.0,NaN,4.66,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5.397605e-79,5.397605e-79,5.000000e+00
1,109264.0,13.0,2.0,1.0,NaN,0.83,17.6,63.8,109.0,109.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5.397605e-79,1.000000e+00,1.000000e+00
2,109265.0,2.0,1.0,3.0,NaN,3.06,15.0,41.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.000000e+00,5.397605e-79,2.000000e+00
3,109266.0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,99.0,99.0,...,30.0,480.0,1.0,10.0,1.0,3.0,7.0,5.397605e-79,5.397605e-79,5.000000e+00
4,109267.0,21.0,2.0,2.0,4.0,5.00,NaN,NaN,NaN,NaN,...,NaN,540.0,NaN,NaN,NaN,1.0,4.0,5.397605e-79,5.397605e-79,5.397605e-79


In [3]:
# Data clean 

df_clean = df_project.copy()
df_clean = df_clean[df_clean["RIDAGEYR"] >= 20].copy()
print(df_clean.shape)
df_clean.head()

(9232, 35)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPXOSY1,BPXOSY2,...,PAD675,PAD680,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910
3,109266.0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,99.0,99.0,...,30.0,480.0,1.0,1.000000e+01,1.0,3.0,7.000000e+00,5.397605e-79,5.397605e-79,5.000000e+00
4,109267.0,21.0,2.0,2.0,4.0,5.00,NaN,NaN,NaN,NaN,...,NaN,540.0,NaN,NaN,NaN,1.0,4.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79
8,109271.0,49.0,1.0,3.0,2.0,NaN,29.7,120.4,102.0,108.0,...,NaN,60.0,1.0,5.397605e-79,NaN,3.0,2.000000e+00,2.000000e+00,5.397605e-79,5.397605e-79
10,109273.0,36.0,1.0,3.0,4.0,0.83,21.9,86.8,116.0,110.0,...,120.0,180.0,1.0,5.397605e-79,NaN,4.0,2.000000e+00,2.000000e+00,5.397605e-79,7.000000e+00
11,109274.0,68.0,1.0,7.0,4.0,1.20,30.2,109.6,138.0,132.0,...,60.0,300.0,1.0,4.000000e+00,2.0,2.0,5.397605e-79,NaN,5.397605e-79,5.397605e-79


In [8]:
# Fix near-zero values that should represent zero
df_clean.loc[df_clean["ALQ121"].abs() < 1e-10, "ALQ121"] = 0
df_clean.loc[df_clean["DBD895"].abs() < 1e-10, "DBD895"] = 0
df_clean.loc[df_clean["DBD900"].abs() < 1e-10, "DBD900"] = 0
df_clean.loc[df_clean["DBD905"].abs() < 1e-10, "DBD905"] = 0
df_clean.loc[df_clean["DBD910"].abs() < 1e-10, "DBD910"] = 0
# clean the missing value codes 
cols_7_9 = ["DMDEDUC2", "DIQ010", "BPQ020", "BPQ080", "SMQ020",
    "PAQ650", "PAQ665", "ALQ111", "DBQ700"]
for col in cols_7_9:
    df_clean[col] = df_clean[col].replace({7: np.nan, 9: np.nan})

cols_77_99 = ["PAQ655", "PAQ670", "ALQ121"]
for col in cols_77_99:
    df_clean[col] = df_clean[col].replace({77: np.nan, 99: np.nan})

df_clean["ALQ130"] = df_clean["ALQ130"].replace({777: np.nan, 999: np.nan})

cols_7777_9999 = ["PAD660", "PAD675", "PAD680",
    "DBD895", "DBD900", "DBD905", "DBD910"]
for col in cols_7777_9999:
    df_clean[col] = df_clean[col].replace({7777: np.nan, 9999: np.nan})

In [9]:
# Create target variable 

df_clean["doctor_diabetes"] = np.where(
    df_clean["DIQ010"] == 1, 1,
    np.where(df_clean["DIQ010"].isin([2, 3]), 0, np.nan)
)
has_target_info = df_clean[["doctor_diabetes", "LBXGH", "LBXGLU"]].notna().any(axis=1)
diabetes_flag = (
    (df_clean["doctor_diabetes"] == 1) |
    (df_clean["LBXGH"] >= 6.5) |
    (df_clean["LBXGLU"] >= 126)
)
df_clean["diabetes"] = np.where(has_target_info, diabetes_flag.astype(int), np.nan)
#df_clean.head()
df_clean_before_targetdrop = df_clean.copy()
print(df_clean_before_targetdrop.shape)

#drop participants without target information
df_clean = df_clean.dropna(subset=["diabetes"]).copy()
df_clean["diabetes"] = df_clean["diabetes"].astype(int)
#df_clean.head()
df_clean["diabetes"].value_counts()

(9232, 37)


diabetes
0    7412
1    1820
Name: count, dtype: int64

In [10]:
df_clean_before_targetdrop.to_csv(processed_dir/"clean_project_dataset_before_targetdrop.csv", index=False)
df_clean.to_csv(processed_dir/"clean_project_dataset.csv", index=False)

In [11]:
df_clean.head()

,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPXOSY1,BPXOSY2,...,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910,doctor_diabetes,diabetes
3,109266.0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,99.0,99.0,...,1.0,10.0,1.0,3.0,7.0,0.0,0.0,5.0,0.0,0
4,109267.0,21.0,2.0,2.0,4.0,5.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,4.0,0.0,0.0,0.0,0.0,0
8,109271.0,49.0,1.0,3.0,2.0,NaN,29.7,120.4,102.0,108.0,...,1.0,0.0,NaN,3.0,2.0,2.0,0.0,0.0,0.0,0
10,109273.0,36.0,1.0,3.0,4.0,0.83,21.9,86.8,116.0,110.0,...,1.0,0.0,NaN,4.0,2.0,2.0,0.0,7.0,0.0,0
11,109274.0,68.0,1.0,7.0,4.0,1.20,30.2,109.6,138.0,132.0,...,1.0,4.0,2.0,2.0,0.0,NaN,0.0,0.0,1.0,1
